<a href="https://colab.research.google.com/github/Plabon-Basak/FlyRank_AI/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Plabon-Basak/FlyRank_AI/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Signal checks and my baseline rule

### Lane
I am keeping my ML-03 lane: **ranking/scoring**. The goal is to assign each content page a priority score so a content/SEO reviewer can decide which pages should be reviewed first. The output is a review queue, not an automatic content change.

### Signal 1 — staleness
**Question:** Do older updates appear more often among pages that are observed as declining?

This is flag-linked because the live session's refresh logic uses **staleness** as a reason to review a page. I will bucket `days_since_last_update` and compare the observed declining rate in each bucket.

**Verdict:** The notebook computes one of `CONFIRMED`, `OPPOSITE`, `MIXED`, or `FALSE` from the observed bucket rates. I will not force a positive result.

### Signal 2 — CTR versus position
**Question:** Does CTR generally fall as average search position gets worse?

This is flag-linked to the session's CTR-fix logic. I will show a bucket table with `n`, mean CTR, and median CTR. The data dictionary warns that position tiers need a volume floor, so the table also reports impressions.

**Verdict:** The notebook computes the same four-way verdict from the observed bucket pattern and prints the rule used.

### Baseline rule
I will use one transparent rule:

> **Score a page by `log1p(impressions_90d)` only when it is both stale (`days_since_last_update >= 180`) and visible (`impressions_90d >= 500`).**

The score is zero otherwise. Every row receives exactly one reason code:
- `stale_visible_page` when the rule fires
- `not_selected` otherwise

The action is:
- `REVIEW_REFRESH` when the rule fires
- `MONITOR` otherwise

This keeps the baseline deliberately simple and auditable. It uses only current-snapshot fields and does not use `trend_direction`, `trend_pct`, or any future-window information.


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
OUTPUT_PATH = Path("work/outputs/baseline_action_score.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Expected {DATA_PATH}. Run this notebook from the repository root "
        "or clone/open your FlyRank repo in Colab first."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)

required = {
    "content_id", "client_id", "days_since_last_update", "impressions_90d",
    "ctr", "avg_position", "is_declining_label"
}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Required columns present.")
print("Rows:", len(df))
print("Declining-label base rate:", f"{df['is_declining_label'].mean():.3f}")

# ---------- Signal 1: staleness ----------
stale_bins = [-np.inf, 30, 90, 180, 365, np.inf]
stale_labels = ["0-30", "31-90", "91-180", "181-365", "365+"]
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=stale_bins,
    labels=stale_labels,
    right=True,
)

stale_table = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          declining_rate=("is_declining_label", "mean"),
          median_impressions=("impressions_90d", "median"),
      )
      .reset_index()
)

print("\nSIGNAL 1 — STALENESS")
display(stale_table.round({"declining_rate": 3, "median_impressions": 1}))

def monotonic_verdict(values, direction="up", tolerance=0.02):
    vals = pd.Series(values).dropna().astype(float).tolist()
    if len(vals) < 3:
        return "MIXED"
    diffs = np.diff(vals)
    if direction == "up":
        if np.all(diffs >= -tolerance) and np.any(diffs > tolerance):
            return "CONFIRMED"
        if np.all(diffs <= tolerance) and np.any(diffs < -tolerance):
            return "OPPOSITE"
    else:
        if np.all(diffs <= tolerance) and np.any(diffs < -tolerance):
            return "CONFIRMED"
        if np.all(diffs >= -tolerance) and np.any(diffs > tolerance):
            return "OPPOSITE"
    return "MIXED"

stale_verdict = monotonic_verdict(
    stale_table["declining_rate"].to_numpy(),
    direction="up",
    tolerance=0.02,
)
print("Staleness verdict:", stale_verdict)
print("Verdict rule: declining rate should move upward across older-update buckets; otherwise the result is MIXED/OPPOSITE.")


## 2. Build the ranked queue

The baseline is intentionally a **hand-written rule**, not a fitted model.

For each page:

1. `visible = impressions_90d >= 500`
2. `stale = days_since_last_update >= 180`
3. `score = stale × visible × log1p(impressions_90d)`
4. Assign exactly one reason code.
5. Rank descending by score and write `work/outputs/baseline_action_score.csv`.

The CSV is a generated artifact and should stay out of git, as required by the assignment.


In [ ]:
# ---------- Signal 2: CTR versus position ----------
# Ignore avg_position == 0 because the data dictionary defines it as "no position data".
position_bins = [-np.inf, 3, 10, 20, 50, np.inf]
position_labels = ["top_3", "page_1", "striking", "page_3_5", "deep"]

signal2 = df[df["avg_position"] > 0].copy()
signal2["position_bucket"] = pd.cut(
    signal2["avg_position"],
    bins=position_bins,
    labels=position_labels,
    right=True,
)

ctr_table = (
    signal2.groupby("position_bucket", observed=False)
           .agg(
               n=("content_id", "size"),
               mean_ctr=("ctr", "mean"),
               median_ctr=("ctr", "median"),
               median_impressions=("impressions_90d", "median"),
           )
           .reset_index()
)

print("\nSIGNAL 2 — CTR VS POSITION")
display(ctr_table.round({"mean_ctr": 3, "median_ctr": 3, "median_impressions": 1}))

ctr_verdict = monotonic_verdict(
    ctr_table["mean_ctr"].to_numpy(),
    direction="down",
    tolerance=0.05,
)
print("CTR-vs-position verdict:", ctr_verdict)
print("Verdict rule: mean CTR should move downward as position gets worse; the tolerance avoids calling tiny changes a directional result.")

# ---------- Baseline rule ----------
out = df.copy()

out["visible"] = (out["impressions_90d"] >= 500).astype(int)
out["stale"] = (out["days_since_last_update"] >= 180).astype(int)

out["baseline_action_score"] = (
    out["stale"] * out["visible"] * np.log1p(out["impressions_90d"])
)

out["reason_code"] = np.where(
    out["baseline_action_score"] > 0,
    "stale_visible_page",
    "not_selected",
)

out["action"] = np.where(
    out["baseline_action_score"] > 0,
    "REVIEW_REFRESH",
    "MONITOR",
)

out["baseline_rank"] = (
    out["baseline_action_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

output_columns = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_action_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position",
]

queue = (
    out[output_columns]
    .sort_values(["baseline_rank", "content_id"])
    .reset_index(drop=True)
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(OUTPUT_PATH, index=False)

print("\nQUEUE WRITTEN")
print("Path:", OUTPUT_PATH)
print("Rows:", len(queue))
print("Selected for review:", int((queue["action"] == "REVIEW_REFRESH").sum()))
print("Top score:", round(float(queue["baseline_action_score"].max()), 3))
display(queue.head(10))


## 3. Top-20 review

The queue is a decision-support list. For each of the top 20 rows, I record:

- **action** — what the rule recommends
- **reason code** — why the page received its score
- **confidence note** — how directly the observed signals support the rule
- **what would make it wrong** — a concrete condition that could make the recommendation inappropriate

This is deliberately a skeptical review. A high score is not proof that a page needs a refresh.


In [ ]:
# Build the required hand-review table from the generated queue.
top20 = queue.head(20).copy()

def confidence_note(row):
    if row["reason_code"] == "stale_visible_page":
        return (
            f"Moderate: the page is {row['days_since_last_update']:.0f} days since update "
            f"and has {row['impressions_90d']:.0f} impressions in the 90-day window."
        )
    return "Low: the baseline did not identify both stale and visible conditions."

def wrong_if(row):
    if row["reason_code"] == "stale_visible_page":
        return (
            "Wrong if the page is intentionally evergreen, the impressions are not a useful "
            "business opportunity, or the reviewer finds no content issue worth refreshing."
        )
    return (
        "Wrong if the page actually has a meaningful refresh opportunity that this simple "
        "staleness-and-visibility rule does not capture."
    )

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(wrong_if, axis=1)

review_columns = [
    "baseline_rank", "content_id", "action", "reason_code",
    "baseline_action_score", "confidence_note", "what_would_make_it_wrong"
]

print("TOP-20 REVIEW")
display(top20[review_columns])

print("\nRequired top-10 review:")
display(top20.head(10)[review_columns])


## 4. Weak picks + leakage check

### Weak picks
The baseline is expected to have weak picks. A stale, high-impression page can still be a bad refresh candidate because the rule does not know the page's business importance, content quality, seasonality, or whether a refresh is already planned.

I will identify the weakest-looking selected rows from the top 20 rather than pretending every top-ranked item is correct.

### Leakage check
The baseline must not use `trend_direction`, `trend_pct`, or any future-window/label-derived field as an input.

The generated queue keeps `is_declining_label` out of the score calculation. It is also not used to decide the reason code or action. The signal audits may compare against the observed label for diagnosis, but that comparison is not an input to the baseline.


In [ ]:
# ---------- Weak-pick review ----------
selected = top20[top20["action"] == "REVIEW_REFRESH"].copy()

if selected.empty:
    print("No selected pages in the top 20; the rule produced no review candidates.")
else:
    weak_picks = selected.sort_values(
        ["baseline_action_score", "impressions_90d"],
        ascending=[True, True],
    ).head(min(3, len(selected)))

    print("WEAK PICKS — lowest-scoring selected rows in the top 20")
    display(
        weak_picks[
            [
                "baseline_rank", "content_id", "baseline_action_score",
                "days_since_last_update", "impressions_90d",
                "reason_code", "action"
            ]
        ]
    )

# ---------- Leakage guard ----------
forbidden_inputs = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
}

score_input_columns = {
    "days_since_last_update",
    "impressions_90d",
}

assert forbidden_inputs.isdisjoint(score_input_columns)

# Explicitly check that the score is reproducible from only the two rule inputs.
recomputed_score = (
    (out["days_since_last_update"] >= 180).astype(int)
    * (out["impressions_90d"] >= 500).astype(int)
    * np.log1p(out["impressions_90d"])
)

assert np.allclose(
    out["baseline_action_score"].to_numpy(),
    recomputed_score.to_numpy(),
    equal_nan=True,
)

print("\nLEAKAGE CHECK")
print("Score inputs:", sorted(score_input_columns))
print("Forbidden label/future fields used as score inputs:", sorted(forbidden_inputs & score_input_columns))
print("Leakage check: PASS")


## Self-check

Before submitting, confirm each line honestly:

- [ ] Two signal checks have visible bucket tables with `n`.
- [ ] At least one checked signal is linked to a FlyRank session flag.
- [ ] Each signal has a one-word verdict: `CONFIRMED`, `OPPOSITE`, `MIXED`, or `FALSE`.
- [ ] The baseline is one transparent rule with a numeric score, exactly one reason code, and an action label.
- [ ] The notebook writes `work/outputs/baseline_action_score.csv`.
- [ ] The top 10 have been reviewed with action, why they are there, and what would make each one wrong.
- [ ] At least one weak pick has been discussed.
- [ ] No `trend_direction`, `trend_pct`, or other label-derived/future-window field is used as a score input.
- [ ] No client names, URLs, or private queries are printed.
- [ ] The notebook runs top to bottom without errors.
- [ ] Commit this notebook under `work/notebooks/w04_baseline_score.ipynb`; keep the generated CSV out of git.
